In [ ]:
import os
import sys
import gc
import time
import hashlib
import threading
from datetime import datetime, timedelta, date
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from numba import njit
import requests

In [ ]:
# Конфиг. Формулы: Документация.md
N_TICKERS = 500
WAIT_TIME = 10
BATCH_SIZE = 500
MAX_BACKOFF = 600

DATA_END_DATE = date(2026, 7, 21)

N_CORES = 15
MC_SCENARIOS = 100_000
GROWTH_THRESHOLD = 0.0025   # Long
SHORT_THRESHOLD = -0.0025   # Short
HISTORY_HOURS = 24
GC_INTERVAL = 10_000
MAX_PLOTS_PER_METHOD = 100000
SPLIT_PLOTS_BY_PNL = True
SHOW_ONLY_PROFITABLE_PLOTS = False
TRADING_HOURS_PER_DAY = 9.5
TRADING_DAYS_PER_YEAR = 252
TRADING_HOURS_PER_YEAR = TRADING_HOURS_PER_DAY * TRADING_DAYS_PER_YEAR
RISK_FREE_RATE_ANNUAL = 0.147
FORECAST_HOURS = 24
LOOKBACK_HOURS = int(TRADING_HOURS_PER_YEAR)
RETRY_BASE_SEC = 1
RETRY_MAX_SEC = 30
RETRY_DELAYS_SEC = [min(RETRY_BASE_SEC * (2 ** k), RETRY_MAX_SEC) for k in range(10)]
VERBOSE_DIAGNOSTICS = True

SIM_TIME = defaultdict(float)
SIM_TIME_LOCK = threading.Lock()

In [ ]:
def fetch_all_tickers(n_tickers: int = N_TICKERS, wait_sec: int = WAIT_TIME,
                          batch_size: int = BATCH_SIZE) -> List[str]:
    """Список тикеров TQBR с MOEX ISS с пагинацией."""
    tickers = []
    start = 0
    i = 0

    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    seen = set()

    while i < n_tickers:
        p = {'securities.columns': 'SECID', 'securities.start': start, 'securities.first': batch_size}

        backoff_delay = 5
        attempt = 0
        success = False

        while backoff_delay <= MAX_BACKOFF and not success:
            try:
                resp = requests.get(url, params=p, timeout=10)
                resp.raise_for_status()
                data = resp.json()

                columns = data['securities']['columns']
                rows = data['securities']['data']

                if not rows:
                    break

                idx_secid = columns.index('SECID')

                for row in rows:
                    if i >= n_tickers:
                        break
                    secid = row[idx_secid]
                    if secid and secid not in seen:
                        seen.add(secid)
                        tickers.append(secid)
                        i += 1
                        print(f"{len(tickers)} {secid}")

                        if i % batch_size == 0:
                            time.sleep(wait_sec)

                start += len(rows)
                success = True

            except Exception as e:
                attempt += 1
                print(f"Ошибка (start={start}, попытка {attempt}): {e}")

                if backoff_delay <= MAX_BACKOFF:
                    print(f"Ожидание {backoff_delay} сек перед повтором...")
                    time.sleep(backoff_delay)
                    backoff_delay *= 2
                else:
                    print(f"Превышено максимальное время ожидания ({MAX_BACKOFF}с). Выход.")
                    break

        if not success:
            break

    return tickers

TICKERS_LIST = fetch_all_tickers(n_tickers=N_TICKERS, wait_sec=WAIT_TIME, batch_size=BATCH_SIZE)
print(f"Загружено тикеров: {len(TICKERS_LIST)}")

In [ ]:
def save_tickers_to_csv(tickers_list: list, filepath: str = 'tickers.csv'):
    """Сохраняет список тикеров в CSV"""
    df = pd.DataFrame({'ticker': tickers_list})
    df.to_csv(filepath, index=False, encoding='utf-8')
    print(f"Сохранено {len(tickers_list)} тикеров в {filepath}")


def load_tickers_from_csv(filepath: str = 'tickers.csv') -> list:
    """Загружает список тикеров из CSV"""
    df = pd.read_csv(filepath)
    tickers_list = df['ticker'].tolist()
    print(f"Загружено {len(tickers_list)} тикеров из {filepath}")
    return tickers_list

In [ ]:
save_tickers_to_csv(TICKERS_LIST, 'tickers.csv')
TICKERS_LIST = load_tickers_from_csv('tickers.csv')
TICKERS_LIST = list(dict.fromkeys(TICKERS_LIST))
len(TICKERS_LIST)

In [ ]:
class LoadingStats:
    """Статистика загрузки и фильтрации."""

    def __init__(self):
        self.error_counts = defaultdict(int)
        self.retry_counts = defaultdict(int)
        self.retry_times = []
        self.successful_tickers = 0
        self.filtered_tickers = defaultdict(list)

    def record_error(self, ticker: str, error_type: str, retries: int, retry_time: float):
        """Учёт ошибки."""
        self.error_counts[error_type] += 1
        self.retry_counts[ticker] = retries
        if retry_time > 0:
            self.retry_times.append(retry_time)

    def record_success(self):
        """Учёт успешного отбора."""
        self.successful_tickers += 1

    def record_filtered(self, ticker: str, reason: str):
        """Учёт отфильтрованного."""
        self.filtered_tickers[reason].append(ticker)

    def get_retry_stats(self) -> dict:
        """Ретраи."""
        if not self.retry_times:
            return {
                'total_retries': 0,
                'avg_retry_time': 0.0,
                'min_retry_time': 0.0,
                'max_retry_time': 0.0,
            }

        return {
            'total_retries': len(self.retry_times),
            'avg_retry_time': float(np.mean(self.retry_times)),
            'min_retry_time': float(np.min(self.retry_times)),
            'max_retry_time': float(np.max(self.retry_times)),
        }

    def get_snapshot(self) -> dict:
        """Снимок."""
        return {
            'successful_tickers': self.successful_tickers,
            'error_counts': dict(self.error_counts),
            'filtered_tickers': {k: list(v) for k, v in self.filtered_tickers.items()},
            'retry_stats': self.get_retry_stats(),
        }


loading_stats = LoadingStats()

# Загрузка MOEX
def _fetch_hourly_data_once(ticker: str, days: int = 365):
    """Один запрос MOEX."""
    end_date = DATA_END_DATE
    start_date = end_date - timedelta(days=days)

    url = (
        "http://iss.moex.com/iss/engines/stock/markets/shares/"
        f"securities/{ticker}/candles.json"
    )
    params = {
        "from": start_date.strftime("%Y-%m-%d"),
        "till": end_date.strftime("%Y-%m-%d"),
        "interval": "60",
        "start": 0,
    }
    all_data = []
    while True:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        if "candles" not in data or not data["candles"]["data"]:
            break

        candles = data["candles"]
        df_chunk = pd.DataFrame(candles["data"], columns=candles["columns"])
        all_data.append(df_chunk)

        if len(candles["data"]) < 500:
            break

        params["start"] += 500
        time.sleep(0.2)

    if not all_data:
        return None, "no_data"

    df = pd.concat(all_data, ignore_index=True)
    del all_data
    gc.collect()

    df["begin"] = pd.to_datetime(df["begin"])
    df = df.set_index("begin").sort_index()
    df = df[["close"]].astype(np.float32)
    df = df[df.index < pd.Timestamp(DATA_END_DATE) + pd.Timedelta(days=1)]

    if len(df) < 120:
        return None, "insufficient_data"

    return df, "success"


def fetch_hourly_data(ticker: str, days: int = 365) -> tuple:
    """Загрузка с ретраями."""
    last_status = "unknown_error"
    retries = 0
    total_retry_time = 0.0

    for attempt, delay in enumerate([0.0] + RETRY_DELAYS_SEC):
        if delay > 0:
            time.sleep(delay)
            total_retry_time += delay
            retries += 1

        try:
            df, status = _fetch_hourly_data_once(ticker, days=days)
            if status in ("no_data", "insufficient_data"):
                loading_stats.record_error(ticker, status, retries, total_retry_time)
                return ticker, None, status, retries, total_retry_time

            if status == "success":
                return ticker, df, "success", retries, total_retry_time

            last_status = status

        except requests.exceptions.Timeout:
            last_status = "timeout"
        except requests.exceptions.RequestException:
            last_status = "api_error"
        except Exception:
            last_status = "unknown_error"

    loading_stats.record_error(ticker, last_status, retries, total_retry_time)
    return ticker, None, last_status, retries, total_retry_time


# GBM: дискретизация в пространстве цен S (Эйлер и Милштейн различаются)
@njit
def generate_scenarios_euler(
    S0: float,
    mu: float,
    sigma: float,
    n_scenarios: int,
    n_steps: int,
    dt: float,
    random_normals: np.ndarray,
) -> np.ndarray:
    """Эйлер–Маруяма в пространстве цен: dS = μS dt + σS dW; сильный порядок 0.5."""
    scenarios = np.zeros((n_scenarios, n_steps + 1), dtype=np.float32)
    scenarios[:, 0] = S0
    sqrt_dt = np.sqrt(dt)
    S_min = S0 * 1e-10

    for i in range(n_steps):
        for j in range(n_scenarios):
            S = scenarios[j, i]
            Z = random_normals[j, i]
            S_new = S + mu * S * dt + sigma * S * sqrt_dt * Z
            if S_new < S_min:
                S_new = S_min
            scenarios[j, i + 1] = S_new

    return scenarios


@njit
def generate_scenarios_milstein(
    S0: float,
    mu: float,
    sigma: float,
    n_scenarios: int,
    n_steps: int,
    dt: float,
    random_normals: np.ndarray,
) -> np.ndarray:
    """Милштейн в пространстве цен: поправка (1/2)σ²S((ΔW)²−Δt); сильный порядок 1."""
    scenarios = np.zeros((n_scenarios, n_steps + 1), dtype=np.float32)
    scenarios[:, 0] = S0
    sqrt_dt = np.sqrt(dt)
    sigma_sq_half = 0.5 * sigma * sigma
    S_min = S0 * 1e-10

    for i in range(n_steps):
        for j in range(n_scenarios):
            S = scenarios[j, i]
            Z = random_normals[j, i]
            dW = sqrt_dt * Z
            S_new = S + mu * S * dt + sigma * S * dW + sigma_sq_half * S * (dW * dW - dt)
            if S_new < S_min:
                S_new = S_min
            scenarios[j, i + 1] = S_new

    return scenarios


def warm_up_numba():
    """Прогрев JIT-компиляции обеих схем до замеров времени."""
    z = np.zeros((1, 1), dtype=np.float32)
    generate_scenarios_euler(1.0, 0.0, 0.1, 1, 1, 1e-4, z)
    generate_scenarios_milstein(1.0, 0.0, 0.1, 1, 1, 1e-4, z)


def ticker_seed(ticker_name: str) -> int:
    """Детерминированный seed генератора по тикеру.

    Parameters:
        ticker_name: тикер MOEX.

    Returns:
        Целое в диапазоне [0, 2^32).
    """
    digest = hashlib.sha256(ticker_name.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little") % (2 ** 32)


def simulate_scenarios(
    method: str,
    S0: float,
    mu: float,
    sigma: float,
    n_scenarios: int,
    random_normals: np.ndarray,
) -> np.ndarray:
    """Симуляция сценариев одной схемой с накоплением чистого времени счёта.

    Parameters:
        method: 'milstein' или 'euler'.
        S0: стартовая цена.
        mu: годовой дрифт.
        sigma: годовая волатильность.
        n_scenarios: число сценариев.
        random_normals: массив N(0,1) формы (n_scenarios, n_steps).

    Returns:
        Массив сценариев формы (n_scenarios, n_steps + 1).
    """
    n_steps = random_normals.shape[1]
    dt = 1.0 / TRADING_HOURS_PER_YEAR
    t0 = time.perf_counter()
    if method == "milstein":
        scenarios = generate_scenarios_milstein(
            S0, mu, sigma, n_scenarios, n_steps, dt, random_normals
        )
    elif method == "euler":
        scenarios = generate_scenarios_euler(
            S0, mu, sigma, n_scenarios, n_steps, dt, random_normals
        )
    else:
        raise ValueError(f"Unknown method: {method}")
    elapsed = time.perf_counter() - t0
    with SIM_TIME_LOCK:
        SIM_TIME[method] += elapsed
    return scenarios


def estimate_parameters(log_returns: np.ndarray):
    """Годовые μ, σ из часовых лог-доходностей.

    Parameters:
        log_returns: часовые лог-доходности обучающего окна.

    Returns:
        (mu, sigma, stats_dict): годовые дрифт и волатильность, служебные оценки.
    """
    mu_hourly = np.mean(log_returns)
    sigma_hourly = np.std(log_returns, ddof=1)

    sigma_annual = sigma_hourly * np.sqrt(TRADING_HOURS_PER_YEAR)
    mu_annual = mu_hourly * TRADING_HOURS_PER_YEAR + 0.5 * (sigma_annual ** 2)

    stats_dict = {
        "n_observations": len(log_returns),
        "mu_annual": float(mu_annual),
        "sigma_annual": float(sigma_annual),
    }

    return float(mu_annual), float(sigma_annual), stats_dict


def validate_parameters(mu: float, sigma: float) -> dict:
    """Проверка оценок μ, σ.

    Parameters:
        mu: годовой дрифт.
        sigma: годовая волатильность.

    Returns:
        dict с полями is_valid и warnings.
    """
    validation = {"is_valid": True, "warnings": []}

    if sigma < 1e-6:
        validation["is_valid"] = False
        validation["warnings"].append("Слишком низкая волатильность (< 1e-6).")
    elif sigma > 5.0:
        validation["warnings"].append(f"Очень высокая волатильность: {sigma:.2%}")

    return validation


# Метрики сценариев (VaR, CVaR, Sortino, Profit Factor, Q₁/Q₉₉)
def evaluate_forecast(
    scenarios: np.ndarray,
    actual_prices: np.ndarray,
    initial_price: float,
) -> dict:
    """Метрики прогнозного распределения и реализованный исход.

    Parameters:
        scenarios: сценарии формы (N, H + 1), включая стартовую цену.
        actual_prices: реализованные цены тестовой выборки.
        initial_price: стартовая цена S0.

    Returns:
        dict метрик для длинной и короткой позиции.
    """
    scenarios_forecast = scenarios[:, 1:]

    metrics = {}
    S0 = float(initial_price)

    median_path = np.median(scenarios_forecast, axis=0)
    median_final = float(median_path[-1])
    median_return = float((median_final - S0) / S0)
    metrics["median_return"] = median_return
    metrics["final_price_median"] = median_final
    metrics["initial_price"] = S0
    n_steps = scenarios_forecast.shape[1]
    actual_exit_idx = min(n_steps, len(actual_prices)) - 1
    metrics["actual_final_price"] = float(actual_prices[actual_exit_idx]) if actual_exit_idx >= 0 and len(actual_prices) > 0 else S0

    final_prices = scenarios_forecast[:, -1]
    scenario_returns = (final_prices - S0) / S0

    metrics["mean_scenario_return"] = float(np.mean(scenario_returns))
    metrics["std_scenario_return"] = float(np.std(scenario_returns))

    var_95_price = np.percentile(final_prices, 5)
    var_95_return = float((var_95_price - S0) / S0)
    metrics["var_95_return"] = var_95_return

    worst_5_percent = final_prices[final_prices <= var_95_price]
    if len(worst_5_percent) > 0:
        cvar_return = float(np.mean((worst_5_percent - S0) / S0))
    else:
        cvar_return = var_95_return
    metrics["cvar_95_return"] = cvar_return

    metrics["max_loss"] = float(np.min(scenario_returns))
    metrics["max_gain"] = float(np.max(scenario_returns))
    metrics["prob_profit"] = float(np.mean(scenario_returns > 0))

    period_years = FORECAST_HOURS / TRADING_HOURS_PER_YEAR
    mar = RISK_FREE_RATE_ANNUAL * period_years
    downside = np.minimum(scenario_returns - mar, 0.0)
    downside_std = float(np.sqrt(np.mean(downside ** 2)))
    metrics["sortino_ratio"] = float((np.mean(scenario_returns) - mar) / (downside_std + 1e-6))

    winning_returns = scenario_returns[scenario_returns > 0]
    losing_returns = np.abs(scenario_returns[scenario_returns < 0])
    sum_wins = float(np.sum(winning_returns)) if len(winning_returns) > 0 else 1e-6
    sum_losses = float(np.sum(losing_returns)) if len(losing_returns) > 0 else 1e-6
    metrics["profit_factor"] = sum_wins / (sum_losses + 1e-6)

    metrics["quantile_01"] = float(np.percentile(final_prices, 1))
    metrics["quantile_99"] = float(np.percentile(final_prices, 99))

    scenario_returns_short = (S0 - final_prices) / S0
    var_95_price_short = np.percentile(final_prices, 95)
    metrics["var_95_return_short"] = float((S0 - var_95_price_short) / S0)
    worst_5pct_short = final_prices[final_prices >= var_95_price_short]
    if len(worst_5pct_short) > 0:
        metrics["cvar_95_return_short"] = float(np.mean((S0 - worst_5pct_short) / S0))
    else:
        metrics["cvar_95_return_short"] = metrics["var_95_return_short"]
    metrics["max_loss_short"] = float(np.min(scenario_returns_short))
    metrics["max_gain_short"] = float(np.max(scenario_returns_short))
    metrics["prob_profit_short"] = float(np.mean(scenario_returns_short > 0))
    metrics["std_scenario_return_short"] = float(np.std(scenario_returns_short))
    downside_short = np.minimum(scenario_returns_short - mar, 0.0)
    downside_std_short = float(np.sqrt(np.mean(downside_short ** 2)))
    metrics["sortino_ratio_short"] = float((np.mean(scenario_returns_short) - mar) / (downside_std_short + 1e-6))
    wins_s = scenario_returns_short[scenario_returns_short > 0]
    loss_s = np.abs(scenario_returns_short[scenario_returns_short < 0])
    sum_wins_s = float(np.sum(wins_s)) if len(wins_s) > 0 else 1e-6
    sum_loss_s = float(np.sum(loss_s)) if len(loss_s) > 0 else 1e-6
    metrics["profit_factor_short"] = float(sum_wins_s / (sum_loss_s + 1e-6))

    del median_path, final_prices, scenario_returns, scenario_returns_short
    del worst_5_percent, worst_5pct_short, downside, downside_short, winning_returns, losing_returns, wins_s, loss_s

    return metrics


# Анализ тикера (train/test, MC, метрики)
def analyze_ticker_both(
    ticker_name: str,
    df: pd.DataFrame,
    n_scenarios: int = MC_SCENARIOS,
) -> dict:
    """Тикер: train/test, μ/σ, обе схемы на одних данных и одних Z.

    Parameters:
        ticker_name: тикер MOEX.
        df: DataFrame часовых цен закрытия.
        n_scenarios: число сценариев Монте-Карло.

    Returns:
        dict {'milstein': result, 'euler': result} либо None, если тикер отфильтрован.
    """
    if len(df) < LOOKBACK_HOURS + FORECAST_HOURS:
        loading_stats.record_filtered(
            ticker_name,
            f"insufficient_history (имеет {len(df)}, нужно ≥{LOOKBACK_HOURS + FORECAST_HOURS})",
        )
        return None

    prices = df["close"].values.astype(np.float32)

    train_prices = prices[-(LOOKBACK_HOURS + FORECAST_HOURS): -FORECAST_HOURS].copy()
    test_prices = prices[-FORECAST_HOURS:].copy()

    del prices

    log_returns_train = np.diff(np.log(train_prices))

    mu, sigma, stats_dict = estimate_parameters(log_returns_train)
    validation = validate_parameters(mu, sigma)

    if not validation["is_valid"]:
        loading_stats.record_filtered(
            ticker_name, f"invalid_parameters (σ={sigma:.6f})"
        )
        del log_returns_train, train_prices, test_prices
        return None

    S0 = float(train_prices[-1])
    rng = np.random.default_rng(ticker_seed(ticker_name))
    random_normals = rng.standard_normal((n_scenarios, FORECAST_HOURS)).astype(np.float32)

    pair = {}
    for method in ("milstein", "euler"):
        scenarios = simulate_scenarios(method, S0, mu, sigma, n_scenarios, random_normals)

        assert np.all(np.isfinite(scenarios)), "Non-finite prices in scenarios"
        assert np.all(scenarios > 0), "Non-positive prices in scenarios"

        forecast_metrics = evaluate_forecast(scenarios, test_prices, S0)

        pair[method] = {
            "ticker": ticker_name,
            "train_prices": train_prices,
            "test_prices": test_prices,
            "scenarios": None,
            "mu": mu,
            "sigma": sigma,
            "n_scenarios": n_scenarios,
            "method": method,
            "stats": stats_dict,
            "metrics": forecast_metrics,
            "current_price": S0,
        }

        del scenarios

    del random_normals, log_returns_train
    loading_stats.record_success()

    return pair


def regenerate_scenarios(mc_result: dict, n_scenarios: int = MC_SCENARIOS) -> np.ndarray:
    """Повторная симуляция сценариев тикера из сохранённых обучающих цен.

    Parameters:
        mc_result: результат одной схемы из analyze_ticker_both.
        n_scenarios: число сценариев.

    Returns:
        Массив сценариев (n_scenarios, H + 1) на тех же данных и тех же Z.
    """
    S0 = float(mc_result["train_prices"][-1])
    rng = np.random.default_rng(ticker_seed(mc_result["ticker"]))
    random_normals = rng.standard_normal((n_scenarios, FORECAST_HOURS)).astype(np.float32)
    scenarios = simulate_scenarios(
        mc_result["method"], S0, mc_result["mu"], mc_result["sigma"], n_scenarios, random_normals
    )
    del random_normals
    return scenarios


def process_and_analyze(
    ticker: str,
    n_scenarios: int = MC_SCENARIOS,
) -> tuple:
    """Загрузка тикера и расчёт обеих схем на одном снимке данных.

    Parameters:
        ticker: тикер MOEX.
        n_scenarios: число сценариев Монте-Карло.

    Returns:
        (ticker, pair, status, retries, retry_time).
    """
    ticker_name, df, status, retries, retry_time = fetch_hourly_data(ticker)

    if status == "success":
        pair = analyze_ticker_both(ticker_name, df, n_scenarios=n_scenarios)
        del df
        return ticker_name, pair, status, retries, retry_time
    else:
        return ticker_name, None, status, retries, retry_time


def print_progress_line(processed: int, total: int, ok: int, errors_count: int):
    """Строка прогресса."""
    line = (
        f"\rОбработано: {processed}/{total} | "
        f"Пригодно: {ok} | Ошибки загрузки: {errors_count}"
    )
    sys.stdout.write(line)
    sys.stdout.flush()


# Портфель: табл.1 — капитал, табл.2 — 1 руб/тикер
def calculate_portfolio_metrics(results_list: list, use_short: bool = False) -> dict:
    """Агрегация по тикерам; P/L по факту; бэктест Q₁/Q₉₉.

    Parameters:
        results_list: результаты одной схемы (Long- или Short-набор).
        use_short: использовать метрики короткой позиции.

    Returns:
        dict метрик портфеля по капиталу и по 1 руб на тикер.
    """
    if not results_list:
        return {}

    n_tickers = len(results_list)
    portfolio_metrics = {"n_tickers": n_tickers}
    current_prices = []
    actual_final_prices = []
    quantile_01_list = []
    quantile_99_list = []
    var_95_returns = []
    cvar_95_returns = []
    max_loss_returns = []
    max_gain_returns = []
    prob_profits = []
    sortino_ratios = []
    profit_factors = []
    std_scenario_returns = []

    for result in results_list:
        m = result["metrics"]
        current_prices.append(m["initial_price"])
        actual_final_prices.append(m["actual_final_price"])
        quantile_01_list.append(m.get("quantile_01", m["initial_price"]))
        quantile_99_list.append(m.get("quantile_99", m["initial_price"]))
        if use_short:
            var_95_returns.append(m["var_95_return_short"])
            cvar_95_returns.append(m["cvar_95_return_short"])
            max_loss_returns.append(m["max_loss_short"])
            max_gain_returns.append(m["max_gain_short"])
            prob_profits.append(m["prob_profit_short"])
            sortino_ratios.append(m["sortino_ratio_short"])
            profit_factors.append(m["profit_factor_short"])
            std_scenario_returns.append(m["std_scenario_return_short"])
        else:
            var_95_returns.append(m["var_95_return"])
            cvar_95_returns.append(m["cvar_95_return"])
            max_loss_returns.append(m["max_loss"])
            max_gain_returns.append(m["max_gain"])
            prob_profits.append(m["prob_profit"])
            sortino_ratios.append(m["sortino_ratio"])
            profit_factors.append(m["profit_factor"])
            std_scenario_returns.append(m["std_scenario_return"])

    current_prices = np.array(current_prices)
    actual_final_prices = np.array(actual_final_prices)
    quantile_01_arr = np.array(quantile_01_list)
    quantile_99_arr = np.array(quantile_99_list)
    var_95_returns = np.array(var_95_returns)
    cvar_95_returns = np.array(cvar_95_returns)
    max_loss_returns = np.array(max_loss_returns)
    max_gain_returns = np.array(max_gain_returns)
    prob_profits = np.array(prob_profits)
    sortino_ratios = np.array(sortino_ratios)
    profit_factors = np.array(profit_factors)
    std_scenario_returns = np.array(std_scenario_returns)

    period_years = FORECAST_HOURS / TRADING_HOURS_PER_YEAR
    total_capital_real = np.sum(current_prices)
    total_final_real = np.sum(actual_final_prices)
    if use_short:
        total_gain_rubles_real = total_capital_real - total_final_real
    else:
        total_gain_rubles_real = total_final_real - total_capital_real
    real_return_pct = (
        total_gain_rubles_real / total_capital_real if total_capital_real > 0 else 0.0
    )

    below_01 = actual_final_prices < quantile_01_arr
    above_99 = actual_final_prices > quantile_99_arr
    count_below_01 = int(np.sum(below_01))
    count_above_99 = int(np.sum(above_99))
    rub_below_01_real = float(np.sum(np.where(below_01, quantile_01_arr - actual_final_prices, 0.0)))
    rub_above_99_real = float(np.sum(np.where(above_99, actual_final_prices - quantile_99_arr, 0.0)))
    rub_below_01_1rub = float(np.sum(np.where(below_01, (quantile_01_arr - actual_final_prices) / current_prices, 0.0)))
    rub_above_99_1rub = float(np.sum(np.where(above_99, (actual_final_prices - quantile_99_arr) / current_prices, 0.0)))

    if total_capital_real > 0:
        weights = current_prices / total_capital_real
    else:
        weights = np.ones_like(current_prices) / len(current_prices)
    weighted_var_95_real = np.sum(var_95_returns * weights)
    weighted_cvar_95_real = np.sum(cvar_95_returns * weights)
    weighted_max_loss_real = np.sum(max_loss_returns * weights)
    weighted_max_gain_real = np.sum(max_gain_returns * weights)
    weighted_prob_profit_real = np.sum(prob_profits * weights)
    weighted_sortino_real = np.sum(sortino_ratios * weights)
    weighted_profit_factor_real = np.sum(profit_factors * weights)

    if use_short:
        individual_returns = (current_prices - actual_final_prices) / current_prices
    else:
        individual_returns = (actual_final_prices - current_prices) / current_prices
    weighted_vol_real = np.sum(std_scenario_returns * weights)
    annualized_vol_real = (
        weighted_vol_real / np.sqrt(period_years) if period_years > 0 else 0.0
    )

    if period_years <= 0:
        sharpe_real = 0.0
    else:
        annualized_return = (1.0 + real_return_pct) ** (1.0 / period_years) - 1.0
        sharpe_real = (
            (annualized_return - RISK_FREE_RATE_ANNUAL) / (annualized_vol_real + 1e-8)
            if annualized_vol_real > 0
            else 0.0
        )

    portfolio_metrics.update(
        {
            "real_price_return_pct": real_return_pct,
            "real_price_gain_rubles": total_gain_rubles_real,
            "real_price_var_95_pct": weighted_var_95_real,
            "real_price_cvar_95_pct": weighted_cvar_95_real,
            "real_price_max_loss_pct": weighted_max_loss_real,
            "real_price_max_gain_pct": weighted_max_gain_real,
            "real_price_prob_profit": weighted_prob_profit_real,
            "real_price_sortino": weighted_sortino_real,
            "real_price_profit_factor": weighted_profit_factor_real,
            "real_price_sharpe_ratio": sharpe_real,
            "total_capital": total_capital_real,
            "count_below_01": count_below_01,
            "count_above_99": count_above_99,
            "count_outside_01_99": count_below_01 + count_above_99,
            "rub_outside_01_real": rub_below_01_real,
            "rub_outside_99_real": rub_above_99_real,
            "rub_outside_01_99_real": rub_below_01_real + rub_above_99_real,
        }
    )

    total_capital_equal_price = float(n_tickers)
    equal_price_return_pct = float(np.mean(individual_returns))
    equal_price_gain_rubles = total_capital_equal_price * equal_price_return_pct
    equal_price_var_95_pct = float(np.mean(var_95_returns))
    equal_price_cvar_95_pct = float(np.mean(cvar_95_returns))

    avg_vol_1rub = float(np.mean(std_scenario_returns))
    annualized_vol_1rub = avg_vol_1rub / np.sqrt(period_years) if period_years > 0 else 0.0

    if period_years <= 0:
        equal_price_sharpe = 0.0
    else:
        annualized_return_ep = (1.0 + equal_price_return_pct) ** (1.0 / period_years) - 1.0
        equal_price_sharpe = (
            (annualized_return_ep - RISK_FREE_RATE_ANNUAL) / (annualized_vol_1rub + 1e-8)
            if annualized_vol_1rub > 0
            else 0.0
        )

    portfolio_metrics.update(
        {
            "equal_price_1rub_return_pct": equal_price_return_pct,
            "equal_price_1rub_gain_rubles": equal_price_gain_rubles,
            "equal_price_1rub_var_95_pct": equal_price_var_95_pct,
            "equal_price_1rub_cvar_95_pct": equal_price_cvar_95_pct,
            "equal_price_1rub_sharpe_ratio": equal_price_sharpe,
            "total_capital_equal_price": total_capital_equal_price,
            "rub_outside_01_1rub": rub_below_01_1rub,
            "rub_outside_99_1rub": rub_above_99_1rub,
            "rub_outside_01_99_1rub": rub_below_01_1rub + rub_above_99_1rub,
        }
    )

    del (
        current_prices,
        actual_final_prices,
        quantile_01_arr,
        quantile_99_arr,
        var_95_returns,
        cvar_95_returns,
        max_loss_returns,
        max_gain_returns,
        prob_profits,
        weights,
        individual_returns,
        std_scenario_returns,
        sortino_ratios,
        profit_factors,
    )

    return portfolio_metrics


def build_portfolio_tables(
    portfolio_milstein: dict,
    portfolio_euler: dict,
    elapsed_milstein: float,
    elapsed_euler: float,
    portfolio_short_milstein: dict = None,
    portfolio_short_euler: dict = None,
):
    """Таблица 1 — по капиталу, таблица 2 — 1 руб/тикер (блоки ВСЕГО, Long, Short).

    Parameters:
        portfolio_milstein: Long-портфель Милштейна.
        portfolio_euler: Long-портфель Эйлера.
        elapsed_milstein: чистое время симуляции Милштейна, сек.
        elapsed_euler: чистое время симуляции Эйлера, сек.
        portfolio_short_milstein: Short-портфель Милштейна.
        portfolio_short_euler: Short-портфель Эйлера.

    Returns:
        (df_table1, df_table2) либо (None, None).
    """
    if not portfolio_milstein or not portfolio_euler:
        return None, None

    n_long_m = safe_get(portfolio_milstein, "n_tickers", 0)
    n_long_e = safe_get(portfolio_euler, "n_tickers", 0)
    n_short_m = safe_get(portfolio_short_milstein, "n_tickers", 0) if portfolio_short_milstein else 0
    n_short_e = safe_get(portfolio_short_euler, "n_tickers", 0) if portfolio_short_euler else 0
    metrics_list = []
    milstein_list = []
    euler_list = []

    cap_total_m = safe_get(portfolio_milstein, "total_capital", 0) + (safe_get(portfolio_short_milstein, "total_capital", 0) if portfolio_short_milstein else 0)
    cap_total_e = safe_get(portfolio_euler, "total_capital", 0) + (safe_get(portfolio_short_euler, "total_capital", 0) if portfolio_short_euler else 0)
    gain_cap_m = safe_get(portfolio_milstein, "real_price_gain_rubles", 0) + (safe_get(portfolio_short_milstein, "real_price_gain_rubles", 0) if portfolio_short_milstein else 0)
    gain_cap_e = safe_get(portfolio_euler, "real_price_gain_rubles", 0) + (safe_get(portfolio_short_euler, "real_price_gain_rubles", 0) if portfolio_short_euler else 0)
    return_cap_m = gain_cap_m / cap_total_m if cap_total_m > 0 else 0.0
    return_cap_e = gain_cap_e / cap_total_e if cap_total_e > 0 else 0.0
    cap_long_m = safe_get(portfolio_milstein, "total_capital", 0)
    cap_short_m = safe_get(portfolio_short_milstein, "total_capital", 0) if portfolio_short_milstein else 0
    cap_long_e = safe_get(portfolio_euler, "total_capital", 0)
    cap_short_e = safe_get(portfolio_short_euler, "total_capital", 0) if portfolio_short_euler else 0

    def _wavg_m(k):
        num = cap_long_m * safe_get(portfolio_milstein, k, 0) + cap_short_m * safe_get(portfolio_short_milstein, k, 0) if portfolio_short_milstein else cap_long_m * safe_get(portfolio_milstein, k, 0)
        return num / cap_total_m if cap_total_m > 0 else 0.0
    def _wavg_e(k):
        num = cap_long_e * safe_get(portfolio_euler, k, 0) + cap_short_e * safe_get(portfolio_short_euler, k, 0) if portfolio_short_euler else cap_long_e * safe_get(portfolio_euler, k, 0)
        return num / cap_total_e if cap_total_e > 0 else 0.0

    count_out_01_99_m = safe_get(portfolio_milstein, "count_outside_01_99", 0) + (safe_get(portfolio_short_milstein, "count_outside_01_99", 0) if portfolio_short_milstein else 0)
    count_out_01_99_e = safe_get(portfolio_euler, "count_outside_01_99", 0) + (safe_get(portfolio_short_euler, "count_outside_01_99", 0) if portfolio_short_euler else 0)
    rub_out_01_99_m = safe_get(portfolio_milstein, "rub_outside_01_99_real", 0) + (safe_get(portfolio_short_milstein, "rub_outside_01_99_real", 0) if portfolio_short_milstein else 0)
    rub_out_01_99_e = safe_get(portfolio_euler, "rub_outside_01_99_real", 0) + (safe_get(portfolio_short_euler, "rub_outside_01_99_real", 0) if portfolio_short_euler else 0)

    metrics_list.append("——— ВСЕГО ———")
    milstein_list.append("—")
    euler_list.append("—")
    metrics_list.append("Кол-во тикеров (Long + Short)")
    milstein_list.append(f"{n_long_m + n_short_m}")
    euler_list.append(f"{n_long_e + n_short_e}")
    metrics_list.append("Общий капитал (РУБ)")
    milstein_list.append(f"{cap_total_m:,.2f}")
    euler_list.append(f"{cap_total_e:,.2f}")
    metrics_list.append("Доходность (%)")
    milstein_list.append(f"{return_cap_m:+.2%}")
    euler_list.append(f"{return_cap_e:+.2%}")
    metrics_list.append("Прибыль (РУБ)")
    milstein_list.append(f"{gain_cap_m:+,.2f}")
    euler_list.append(f"{gain_cap_e:+,.2f}")
    metrics_list.append("VaR (95%) %")
    milstein_list.append(f"{_wavg_m('real_price_var_95_pct'):.2%}")
    euler_list.append(f"{_wavg_e('real_price_var_95_pct'):.2%}")
    metrics_list.append("CVaR (95%) %")
    milstein_list.append(f"{_wavg_m('real_price_cvar_95_pct'):.2%}")
    euler_list.append(f"{_wavg_e('real_price_cvar_95_pct'):.2%}")
    metrics_list.append("Max Loss %")
    milstein_list.append(f"{_wavg_m('real_price_max_loss_pct'):.2%}")
    euler_list.append(f"{_wavg_e('real_price_max_loss_pct'):.2%}")
    metrics_list.append("Max Gain %")
    milstein_list.append(f"{_wavg_m('real_price_max_gain_pct'):.2%}")
    euler_list.append(f"{_wavg_e('real_price_max_gain_pct'):.2%}")
    metrics_list.append("P(profit)")
    milstein_list.append(f"{_wavg_m('real_price_prob_profit'):.1%}")
    euler_list.append(f"{_wavg_e('real_price_prob_profit'):.1%}")
    metrics_list.append("Sortino")
    milstein_list.append(f"{_wavg_m('real_price_sortino'):.4f}")
    euler_list.append(f"{_wavg_e('real_price_sortino'):.4f}")
    metrics_list.append("Profit Factor")
    milstein_list.append(f"{_wavg_m('real_price_profit_factor'):.4f}")
    euler_list.append(f"{_wavg_e('real_price_profit_factor'):.4f}")
    metrics_list.append("Sharpe Ratio")
    milstein_list.append(f"{_wavg_m('real_price_sharpe_ratio'):.4f}")
    euler_list.append(f"{_wavg_e('real_price_sharpe_ratio'):.4f}")
    metrics_list.append("Время симуляции (сек)")
    milstein_list.append(f"{elapsed_milstein:.2f}")
    euler_list.append(f"{elapsed_euler:.2f}")
    metrics_list.append("Кол-во раз вне границ [1%; 99%]")
    milstein_list.append(f"{count_out_01_99_m}")
    euler_list.append(f"{count_out_01_99_e}")
    metrics_list.append("Сумма РУБ вне границ [1%; 99%]")
    milstein_list.append(f"{rub_out_01_99_m:+,.2f}")
    euler_list.append(f"{rub_out_01_99_e:+,.2f}")

    metrics_list.append("——— Long (по весу капитала) ———")
    milstein_list.append("—")
    euler_list.append("—")
    metrics_long = [
        "Кол-во тикеров", "Общий капитал (РУБ)", "Доходность (%)", "Прибыль (РУБ)",
        "VaR (95%) %", "CVaR (95%) %", "Max Loss %", "Max Gain %",
        "P(profit)", "Sortino", "Profit Factor", "Sharpe Ratio",
        "Кол-во раз вне границ [1%; 99%]", "Сумма РУБ вне границ [1%; 99%]"
    ]
    metrics_list.extend(metrics_long)
    milstein_list.extend([
        f"{n_long_m}",
        f"{safe_get(portfolio_milstein, 'total_capital', 0):,.2f}",
        f"{safe_get(portfolio_milstein, 'real_price_return_pct', 0):+.2%}",
        f"{safe_get(portfolio_milstein, 'real_price_gain_rubles', 0):+,.2f}",
        f"{safe_get(portfolio_milstein, 'real_price_var_95_pct', 0):.2%}",
        f"{safe_get(portfolio_milstein, 'real_price_cvar_95_pct', 0):.2%}",
        f"{safe_get(portfolio_milstein, 'real_price_max_loss_pct', 0):.2%}",
        f"{safe_get(portfolio_milstein, 'real_price_max_gain_pct', 0):.2%}",
        f"{safe_get(portfolio_milstein, 'real_price_prob_profit', 0):.1%}",
        f"{safe_get(portfolio_milstein, 'real_price_sortino', 0):.4f}",
        f"{safe_get(portfolio_milstein, 'real_price_profit_factor', 0):.4f}",
        f"{safe_get(portfolio_milstein, 'real_price_sharpe_ratio', 0):.4f}",
        f"{safe_get(portfolio_milstein, 'count_outside_01_99', 0)}",
        f"{safe_get(portfolio_milstein, 'rub_outside_01_99_real', 0):+,.2f}",
    ])
    euler_list.extend([
        f"{n_long_e}",
        f"{safe_get(portfolio_euler, 'total_capital', 0):,.2f}",
        f"{safe_get(portfolio_euler, 'real_price_return_pct', 0):+.2%}",
        f"{safe_get(portfolio_euler, 'real_price_gain_rubles', 0):+,.2f}",
        f"{safe_get(portfolio_euler, 'real_price_var_95_pct', 0):.2%}",
        f"{safe_get(portfolio_euler, 'real_price_cvar_95_pct', 0):.2%}",
        f"{safe_get(portfolio_euler, 'real_price_max_loss_pct', 0):.2%}",
        f"{safe_get(portfolio_euler, 'real_price_max_gain_pct', 0):.2%}",
        f"{safe_get(portfolio_euler, 'real_price_prob_profit', 0):.1%}",
        f"{safe_get(portfolio_euler, 'real_price_sortino', 0):.4f}",
        f"{safe_get(portfolio_euler, 'real_price_profit_factor', 0):.4f}",
        f"{safe_get(portfolio_euler, 'real_price_sharpe_ratio', 0):.4f}",
        f"{safe_get(portfolio_euler, 'count_outside_01_99', 0)}",
        f"{safe_get(portfolio_euler, 'rub_outside_01_99_real', 0):+,.2f}",
    ])

    if portfolio_short_milstein and portfolio_short_euler:
        metrics_list.append("——— Short (по весу капитала) ———")
        milstein_list.append("—")
        euler_list.append("—")
        metrics_short = [
            "Кол-во тикеров", "Общий капитал (РУБ)", "Доходность (%)", "Прибыль (РУБ)",
            "VaR (95%) %", "CVaR (95%) %", "Max Loss %", "Max Gain %",
            "P(profit)", "Sortino", "Profit Factor", "Sharpe Ratio",
            "Кол-во раз вне границ [1%; 99%]", "Сумма РУБ вне границ [1%; 99%]"
        ]
        metrics_list.extend(metrics_short)
        milstein_list.extend([
            f"{n_short_m}",
            f"{safe_get(portfolio_short_milstein, 'total_capital', 0):,.2f}",
            f"{safe_get(portfolio_short_milstein, 'real_price_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_short_milstein, 'real_price_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_short_milstein, 'real_price_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_milstein, 'real_price_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_milstein, 'real_price_max_loss_pct', 0):.2%}",
            f"{safe_get(portfolio_short_milstein, 'real_price_max_gain_pct', 0):.2%}",
            f"{safe_get(portfolio_short_milstein, 'real_price_prob_profit', 0):.1%}",
            f"{safe_get(portfolio_short_milstein, 'real_price_sortino', 0):.4f}",
            f"{safe_get(portfolio_short_milstein, 'real_price_profit_factor', 0):.4f}",
            f"{safe_get(portfolio_short_milstein, 'real_price_sharpe_ratio', 0):.4f}",
            f"{safe_get(portfolio_short_milstein, 'count_outside_01_99', 0)}",
            f"{safe_get(portfolio_short_milstein, 'rub_outside_01_99_real', 0):+,.2f}",
        ])
        euler_list.extend([
            f"{n_short_e}",
            f"{safe_get(portfolio_short_euler, 'total_capital', 0):,.2f}",
            f"{safe_get(portfolio_short_euler, 'real_price_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_short_euler, 'real_price_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_short_euler, 'real_price_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_euler, 'real_price_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_euler, 'real_price_max_loss_pct', 0):.2%}",
            f"{safe_get(portfolio_short_euler, 'real_price_max_gain_pct', 0):.2%}",
            f"{safe_get(portfolio_short_euler, 'real_price_prob_profit', 0):.1%}",
            f"{safe_get(portfolio_short_euler, 'real_price_sortino', 0):.4f}",
            f"{safe_get(portfolio_short_euler, 'real_price_profit_factor', 0):.4f}",
            f"{safe_get(portfolio_short_euler, 'real_price_sharpe_ratio', 0):.4f}",
            f"{safe_get(portfolio_short_euler, 'count_outside_01_99', 0)}",
            f"{safe_get(portfolio_short_euler, 'rub_outside_01_99_real', 0):+,.2f}",
        ])

    data_real = {"Метрика": metrics_list, "Милштейн": milstein_list, "Эйлер": euler_list}
    df_table1 = pd.DataFrame(data_real)
    metrics_ep = []
    milstein_ep = []
    euler_ep = []

    total_n_m = n_long_m + n_short_m
    total_n_e = n_long_e + n_short_e
    gain_m = safe_get(portfolio_milstein, "equal_price_1rub_gain_rubles", 0) + (
        safe_get(portfolio_short_milstein, "equal_price_1rub_gain_rubles", 0) if portfolio_short_milstein else 0
    )
    gain_e = safe_get(portfolio_euler, "equal_price_1rub_gain_rubles", 0) + (
        safe_get(portfolio_short_euler, "equal_price_1rub_gain_rubles", 0) if portfolio_short_euler else 0
    )
    return_total_m = gain_m / total_n_m if total_n_m > 0 else 0.0
    return_total_e = gain_e / total_n_e if total_n_e > 0 else 0.0
    var_total_m = (
        n_long_m * safe_get(portfolio_milstein, "equal_price_1rub_var_95_pct", 0)
        + (n_short_m * safe_get(portfolio_short_milstein, "equal_price_1rub_var_95_pct", 0) if portfolio_short_milstein else 0)
    ) / total_n_m if total_n_m > 0 else 0.0
    var_total_e = (
        n_long_e * safe_get(portfolio_euler, "equal_price_1rub_var_95_pct", 0)
        + (n_short_e * safe_get(portfolio_short_euler, "equal_price_1rub_var_95_pct", 0) if portfolio_short_euler else 0)
    ) / total_n_e if total_n_e > 0 else 0.0
    cvar_total_m = (
        n_long_m * safe_get(portfolio_milstein, "equal_price_1rub_cvar_95_pct", 0)
        + (n_short_m * safe_get(portfolio_short_milstein, "equal_price_1rub_cvar_95_pct", 0) if portfolio_short_milstein else 0)
    ) / total_n_m if total_n_m > 0 else 0.0
    cvar_total_e = (
        n_long_e * safe_get(portfolio_euler, "equal_price_1rub_cvar_95_pct", 0)
        + (n_short_e * safe_get(portfolio_short_euler, "equal_price_1rub_cvar_95_pct", 0) if portfolio_short_euler else 0)
    ) / total_n_e if total_n_e > 0 else 0.0
    sharpe_total_m = (
        n_long_m * safe_get(portfolio_milstein, "equal_price_1rub_sharpe_ratio", 0)
        + (n_short_m * safe_get(portfolio_short_milstein, "equal_price_1rub_sharpe_ratio", 0) if portfolio_short_milstein else 0)
    ) / total_n_m if total_n_m > 0 else 0.0
    sharpe_total_e = (
        n_long_e * safe_get(portfolio_euler, "equal_price_1rub_sharpe_ratio", 0)
        + (n_short_e * safe_get(portfolio_short_euler, "equal_price_1rub_sharpe_ratio", 0) if portfolio_short_euler else 0)
    ) / total_n_e if total_n_e > 0 else 0.0
    count_below_01_total_m = safe_get(portfolio_milstein, "count_below_01", 0) + (safe_get(portfolio_short_milstein, "count_below_01", 0) if portfolio_short_milstein else 0)
    count_above_99_total_m = safe_get(portfolio_milstein, "count_above_99", 0) + (safe_get(portfolio_short_milstein, "count_above_99", 0) if portfolio_short_milstein else 0)
    rub_01_total_m = safe_get(portfolio_milstein, "rub_outside_01_1rub", 0) + (safe_get(portfolio_short_milstein, "rub_outside_01_1rub", 0) if portfolio_short_milstein else 0)
    rub_99_total_m = safe_get(portfolio_milstein, "rub_outside_99_1rub", 0) + (safe_get(portfolio_short_milstein, "rub_outside_99_1rub", 0) if portfolio_short_milstein else 0)
    count_below_01_total_e = safe_get(portfolio_euler, "count_below_01", 0) + (safe_get(portfolio_short_euler, "count_below_01", 0) if portfolio_short_euler else 0)
    count_above_99_total_e = safe_get(portfolio_euler, "count_above_99", 0) + (safe_get(portfolio_short_euler, "count_above_99", 0) if portfolio_short_euler else 0)
    rub_01_total_e = safe_get(portfolio_euler, "rub_outside_01_1rub", 0) + (safe_get(portfolio_short_euler, "rub_outside_01_1rub", 0) if portfolio_short_euler else 0)
    rub_99_total_e = safe_get(portfolio_euler, "rub_outside_99_1rub", 0) + (safe_get(portfolio_short_euler, "rub_outside_99_1rub", 0) if portfolio_short_euler else 0)

    metrics_ep.append("——— ВСЕГО ———")
    milstein_ep.append("—")
    euler_ep.append("—")
    metrics_ep.append("Кол-во тикеров (Long + Short)")
    milstein_ep.append(f"{total_n_m}")
    euler_ep.append(f"{total_n_e}")
    metrics_ep.extend(["Портфель (РУБ)", "Доходность (%)", "Прибыль (РУБ)", "VaR (95%) %", "CVaR (95%) %", "Sharpe Ratio", "Кол-во раз вне границ [1%; 99%]", "Сумма РУБ вне границ [1%; 99%]"])
    milstein_ep.extend([
        f"{total_n_m:.2f}",
        f"{return_total_m:+.2%}",
        f"{gain_m:+,.2f}",
        f"{var_total_m:.2%}",
        f"{cvar_total_m:.2%}",
        f"{sharpe_total_m:.4f}",
        f"{count_below_01_total_m + count_above_99_total_m}",
        f"{rub_01_total_m + rub_99_total_m:+,.2f}",
    ])
    euler_ep.extend([
        f"{total_n_e:.2f}",
        f"{return_total_e:+.2%}",
        f"{gain_e:+,.2f}",
        f"{var_total_e:.2%}",
        f"{cvar_total_e:.2%}",
        f"{sharpe_total_e:.4f}",
        f"{count_below_01_total_e + count_above_99_total_e}",
        f"{rub_01_total_e + rub_99_total_e:+,.2f}",
    ])

    metrics_ep.append("——— Long (по 1 руб/тикер) ———")
    milstein_ep.append("—")
    euler_ep.append("—")
    metrics_ep.extend(["Кол-во тикеров", "Портфель (РУБ)", "Доходность (%)", "Прибыль (РУБ)", "VaR (95%) %", "CVaR (95%) %", "Sharpe Ratio", "Кол-во раз вне границ [1%; 99%]", "Сумма РУБ вне границ [1%; 99%]"])
    milstein_ep.extend([
        f"{n_long_m}",
        f"{safe_get(portfolio_milstein, 'total_capital_equal_price', 0):.2f}",
        f"{safe_get(portfolio_milstein, 'equal_price_1rub_return_pct', 0):+.2%}",
        f"{safe_get(portfolio_milstein, 'equal_price_1rub_gain_rubles', 0):+,.2f}",
        f"{safe_get(portfolio_milstein, 'equal_price_1rub_var_95_pct', 0):.2%}",
        f"{safe_get(portfolio_milstein, 'equal_price_1rub_cvar_95_pct', 0):.2%}",
        f"{safe_get(portfolio_milstein, 'equal_price_1rub_sharpe_ratio', 0):.4f}",
        f"{safe_get(portfolio_milstein, 'count_outside_01_99', 0)}",
        f"{safe_get(portfolio_milstein, 'rub_outside_01_99_1rub', 0):+,.2f}",
    ])
    euler_ep.extend([
        f"{n_long_e}",
        f"{safe_get(portfolio_euler, 'total_capital_equal_price', 0):.2f}",
        f"{safe_get(portfolio_euler, 'equal_price_1rub_return_pct', 0):+.2%}",
        f"{safe_get(portfolio_euler, 'equal_price_1rub_gain_rubles', 0):+,.2f}",
        f"{safe_get(portfolio_euler, 'equal_price_1rub_var_95_pct', 0):.2%}",
        f"{safe_get(portfolio_euler, 'equal_price_1rub_cvar_95_pct', 0):.2%}",
        f"{safe_get(portfolio_euler, 'equal_price_1rub_sharpe_ratio', 0):.4f}",
        f"{safe_get(portfolio_euler, 'count_outside_01_99', 0)}",
        f"{safe_get(portfolio_euler, 'rub_outside_01_99_1rub', 0):+,.2f}",
    ])

    if portfolio_short_milstein and portfolio_short_euler:
        metrics_ep.append("——— Short (по 1 руб/тикер) ———")
        milstein_ep.append("—")
        euler_ep.append("—")
        metrics_ep.extend(["Кол-во тикеров", "Портфель (РУБ)", "Доходность (%)", "Прибыль (РУБ)", "VaR (95%) %", "CVaR (95%) %", "Sharpe Ratio", "Кол-во раз вне границ [1%; 99%]", "Сумма РУБ вне границ [1%; 99%]"])
        milstein_ep.extend([
            f"{n_short_m}",
            f"{safe_get(portfolio_short_milstein, 'total_capital_equal_price', 0):.2f}",
            f"{safe_get(portfolio_short_milstein, 'equal_price_1rub_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_short_milstein, 'equal_price_1rub_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_short_milstein, 'equal_price_1rub_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_milstein, 'equal_price_1rub_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_milstein, 'equal_price_1rub_sharpe_ratio', 0):.4f}",
            f"{safe_get(portfolio_short_milstein, 'count_outside_01_99', 0)}",
            f"{safe_get(portfolio_short_milstein, 'rub_outside_01_99_1rub', 0):+,.2f}",
        ])
        euler_ep.extend([
            f"{n_short_e}",
            f"{safe_get(portfolio_short_euler, 'total_capital_equal_price', 0):.2f}",
            f"{safe_get(portfolio_short_euler, 'equal_price_1rub_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_short_euler, 'equal_price_1rub_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_short_euler, 'equal_price_1rub_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_euler, 'equal_price_1rub_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_short_euler, 'equal_price_1rub_sharpe_ratio', 0):.4f}",
            f"{safe_get(portfolio_short_euler, 'count_outside_01_99', 0)}",
            f"{safe_get(portfolio_short_euler, 'rub_outside_01_99_1rub', 0):+,.2f}",
        ])

    data_equal_price = {"Метрика": metrics_ep, "Милштейн": milstein_ep, "Эйлер": euler_ep}
    df_table2 = pd.DataFrame(data_equal_price)
    return df_table1, df_table2


def safe_get(d: dict, key: str, default=0):
    """d[key] или default."""
    if not d:
        return default
    return d.get(key, default)


def print_loading_statistics(snapshot: dict):
    """Статистика загрузки и фильтрации данных.

    Parameters:
        snapshot: снимок LoadingStats.get_snapshot().
    """
    print("\n" + "=" * 100)
    print("СТАТИСТИКА ЗАГРУЗКИ И ОБРАБОТКИ ДАННЫХ")
    print("=" * 100)

    ok = snapshot['successful_tickers']
    ec = snapshot['error_counts']
    filtered = snapshot['filtered_tickers']
    total_errors = sum(ec.values())
    total_filtered = sum(len(v) for v in filtered.values())
    total = ok + total_errors + total_filtered
    if total == 0:
        print("Нет данных")
        print("=" * 100)
        return

    pct = lambda x: 100.0 * x / total
    no_data = ec.get('no_data', 0) + ec.get('insufficient_data', 0)
    connection = ec.get('timeout', 0) + ec.get('api_error', 0)
    other_errors = total_errors - no_data - connection
    print(f"Обработано {total}: пригодно {ok} ({pct(ok):.1f}%), нет данных {pct(no_data):.1f}%, "
          f"ошибка соединения {pct(connection):.1f}%, отфильтровано {pct(total_filtered):.1f}%" +
          (f", прочее {pct(other_errors):.1f}%" if other_errors > 0 else ""))
    retry_stats = snapshot['retry_stats']
    if retry_stats['total_retries'] > 0:
        print(f"Ретраи: {retry_stats['total_retries']}, среднее ожидание {retry_stats['avg_retry_time']:.1f} с")
    print("=" * 100)


def print_portfolio_summary(
    portfolio_milstein: dict,
    portfolio_euler: dict,
    elapsed_milstein: float,
    elapsed_euler: float,
    portfolio_short_milstein: dict = None,
    portfolio_short_euler: dict = None,
):
    """Сводка Long/Short в консоль.

    Parameters:
        portfolio_milstein: Long-портфель Милштейна.
        portfolio_euler: Long-портфель Эйлера.
        elapsed_milstein: чистое время симуляции Милштейна, сек.
        elapsed_euler: чистое время симуляции Эйлера, сек.
        portfolio_short_milstein: Short-портфель Милштейна.
        portfolio_short_euler: Short-портфель Эйлера.
    """
    print("\n" + "=" * 140)
    print("ПОРТФЕЛЬНЫЕ МЕТРИКИ: СРАВНЕНИЕ МЕТОДОВ (Long / Short)")
    print("=" * 140)
    print()

    if not portfolio_milstein or not portfolio_euler:
        print("⚠️ ОШИБКА: Один или оба портфеля пусты!")
        if not portfolio_milstein:
            print(" • Метод Милштейна: не удалось получить данные")
        if not portfolio_euler:
            print(" • Метод Эйлера: не удалось получить данные")
        print()
        print("Возможные причины:")
        print(" 1. Все загруженные тикеры отфильтрованы")
        print(" 2. API MOEX недоступен")
        print(" 3. Тикеры неверные")
        print("=" * 140)
        return

    print("ТАБЛИЦА 1 (блок Long): по капиталу — только тикеры с медианной доходностью ≥ порога; вес = доля капитала")
    print("-" * 140)

    data_real = {
        "Метрика": [
            "Кол-во тикеров", "Общий капитал (РУБ)", "Доходность (%)", "Прибыль (РУБ)",
            "VaR (95%) %", "CVaR (95%) %", "Max Loss %", "Max Gain %",
            "P(profit)", "Sortino", "Profit Factor", "Sharpe Ratio", "Время симуляции (сек)"
        ],
        "Милштейн": [
            f"{safe_get(portfolio_milstein, 'n_tickers', 0)}",
            f"{safe_get(portfolio_milstein, 'total_capital', 0):,.2f}",
            f"{safe_get(portfolio_milstein, 'real_price_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_milstein, 'real_price_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_milstein, 'real_price_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_milstein, 'real_price_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_milstein, 'real_price_max_loss_pct', 0):.2%}",
            f"{safe_get(portfolio_milstein, 'real_price_max_gain_pct', 0):.2%}",
            f"{safe_get(portfolio_milstein, 'real_price_prob_profit', 0):.1%}",
            f"{safe_get(portfolio_milstein, 'real_price_sortino', 0):.4f}",
            f"{safe_get(portfolio_milstein, 'real_price_profit_factor', 0):.4f}",
            f"{safe_get(portfolio_milstein, 'real_price_sharpe_ratio', 0):.4f}",
            f"{elapsed_milstein:.2f}",
        ],
        "Эйлер": [
            f"{safe_get(portfolio_euler, 'n_tickers', 0)}",
            f"{safe_get(portfolio_euler, 'total_capital', 0):,.2f}",
            f"{safe_get(portfolio_euler, 'real_price_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_euler, 'real_price_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_euler, 'real_price_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_euler, 'real_price_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_euler, 'real_price_max_loss_pct', 0):.2%}",
            f"{safe_get(portfolio_euler, 'real_price_max_gain_pct', 0):.2%}",
            f"{safe_get(portfolio_euler, 'real_price_prob_profit', 0):.1%}",
            f"{safe_get(portfolio_euler, 'real_price_sortino', 0):.4f}",
            f"{safe_get(portfolio_euler, 'real_price_profit_factor', 0):.4f}",
            f"{safe_get(portfolio_euler, 'real_price_sharpe_ratio', 0):.4f}",
            f"{elapsed_euler:.2f}",
        ],
    }

    df_real = pd.DataFrame(data_real)
    print(df_real.to_string(index=False))
    print()

    print("ТАБЛИЦА 2: все найденные тикеры по одной штуке, каждая позиция = 1 руб")
    print("-" * 140)

    data_equal_price = {
        "Метрика": [
            "Портфель (РУБ)", "Доходность (%)", "Прибыль (РУБ)",
            "VaR (95%) %", "CVaR (95%) %", "Sharpe Ratio"
        ],
        "Милштейн": [
            f"{safe_get(portfolio_milstein, 'total_capital_equal_price', 0):.2f}",
            f"{safe_get(portfolio_milstein, 'equal_price_1rub_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_milstein, 'equal_price_1rub_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_milstein, 'equal_price_1rub_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_milstein, 'equal_price_1rub_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_milstein, 'equal_price_1rub_sharpe_ratio', 0):.4f}",
        ],
        "Эйлер": [
            f"{safe_get(portfolio_euler, 'total_capital_equal_price', 0):.2f}",
            f"{safe_get(portfolio_euler, 'equal_price_1rub_return_pct', 0):+.2%}",
            f"{safe_get(portfolio_euler, 'equal_price_1rub_gain_rubles', 0):+,.2f}",
            f"{safe_get(portfolio_euler, 'equal_price_1rub_var_95_pct', 0):.2%}",
            f"{safe_get(portfolio_euler, 'equal_price_1rub_cvar_95_pct', 0):.2%}",
            f"{safe_get(portfolio_euler, 'equal_price_1rub_sharpe_ratio', 0):.4f}",
        ],
    }

    df_equal_price = pd.DataFrame(data_equal_price)
    print(df_equal_price.to_string(index=False))
    print()

    if portfolio_short_milstein and portfolio_short_euler:
        print("--- Short (продажа в t=0, выкуп в конце) ---")
        print("-" * 140)
        data_short = {
            "Метрика": ["Кол-во тикеров", "Общий капитал (РУБ)", "Доходность Short (%)", "Прибыль (РУБ)", "Sharpe Ratio"],
            "Милштейн": [
                f"{safe_get(portfolio_short_milstein, 'n_tickers', 0)}",
                f"{safe_get(portfolio_short_milstein, 'total_capital', 0):,.2f}",
                f"{safe_get(portfolio_short_milstein, 'real_price_return_pct', 0):+.2%}",
                f"{safe_get(portfolio_short_milstein, 'real_price_gain_rubles', 0):+,.2f}",
                f"{safe_get(portfolio_short_milstein, 'real_price_sharpe_ratio', 0):.4f}",
            ],
            "Эйлер": [
                f"{safe_get(portfolio_short_euler, 'n_tickers', 0)}",
                f"{safe_get(portfolio_short_euler, 'total_capital', 0):,.2f}",
                f"{safe_get(portfolio_short_euler, 'real_price_return_pct', 0):+.2%}",
                f"{safe_get(portfolio_short_euler, 'real_price_gain_rubles', 0):+,.2f}",
                f"{safe_get(portfolio_short_euler, 'real_price_sharpe_ratio', 0):.4f}",
            ],
        }
        print(pd.DataFrame(data_short).to_string(index=False))
        print()

    print("=" * 140)


# Графики прогноза
def save_mc_forecast_to_file(mc_result: dict, method: str, output_dir: str = ".mcplots"):
    """График квантили/медиана/факт.

    Parameters:
        mc_result: результат одной схемы с заполненным полем scenarios.
        method: 'milstein' или 'euler'.
        output_dir: папка для сохранения.

    Returns:
        Путь к файлу либо None.
    """
    if mc_result["scenarios"] is None:
        return None

    scenarios = mc_result["scenarios"][:, 1:].astype(np.float32)
    n_steps = scenarios.shape[1]

    test_prices = mc_result["test_prices"]
    train_prices = mc_result["train_prices"]
    ticker = mc_result["ticker"]
    metrics = mc_result["metrics"]
    S0 = float(train_prices[-1])
    median_final = float(metrics["final_price_median"])
    mr = float(metrics["median_return"])
    actual_exit_idx = min(n_steps, len(test_prices)) - 1
    actual_exit_price = float(test_prices[actual_exit_idx]) if actual_exit_idx >= 0 else S0
    if mr >= GROWTH_THRESHOLD:
        position_label = "Long (покупка)"
        entry_price = S0
        exit_price_median = median_final
        pnl_median_rub = median_final - S0
        pnl_median_pct = mr
        actual_pnl_rub = actual_exit_price - S0
        actual_pnl_pct = (actual_exit_price - S0) / S0
    elif mr <= SHORT_THRESHOLD:
        position_label = "Short (продажа)"
        entry_price = S0
        exit_price_median = median_final
        pnl_median_rub = S0 - median_final
        pnl_median_pct = (S0 - median_final) / S0
        actual_pnl_rub = S0 - actual_exit_price
        actual_pnl_pct = (S0 - actual_exit_price) / S0
    else:
        position_label = "Вне порога (нейтрально)"
        entry_price = S0
        exit_price_median = median_final
        pnl_median_rub = median_final - S0
        pnl_median_pct = mr
        actual_pnl_rub = actual_exit_price - S0
        actual_pnl_pct = (actual_exit_price - S0) / S0

    if SHOW_ONLY_PROFITABLE_PLOTS and actual_pnl_pct <= 0:
        return None

    if SPLIT_PLOTS_BY_PNL:
        subdir = "pnl_positive" if actual_pnl_pct > 0 else "pnl_non_positive"
        save_dir = os.path.join(output_dir, subdir)
    else:
        save_dir = output_dir
    os.makedirs(save_dir, exist_ok=True)

    max_reasonable = S0 * 100
    scenarios = np.minimum(scenarios, max_reasonable)
    quantile_01 = np.percentile(scenarios, 1, axis=0)
    quantile_05 = np.percentile(scenarios, 5, axis=0)
    quantile_25 = np.percentile(scenarios, 25, axis=0)
    quantile_50 = np.percentile(scenarios, 50, axis=0)
    quantile_75 = np.percentile(scenarios, 75, axis=0)
    quantile_95 = np.percentile(scenarios, 95, axis=0)
    quantile_99 = np.percentile(scenarios, 99, axis=0)

    history_len = min(HISTORY_HOURS, len(train_prices))
    history = np.asarray(train_prices[-history_len:])
    h_time = np.arange(-history_len, 0, dtype=float)
    f_time = np.arange(0, n_steps, dtype=float)

    fig, ax = plt.subplots(figsize=(14, 7))

    ax.plot(h_time, history, color='#1f77b4', linewidth=2, label='История (обучающая выборка)', alpha=0.9, zorder=3)
    n_plot = min(n_steps, len(test_prices))
    ax.plot(f_time[:n_plot], test_prices[:n_plot], color='#2ecc71', linewidth=2, label='Факт (вне выборки)', alpha=0.9, zorder=3)

    n_show = min(80, scenarios.shape[0])
    for i in range(n_show):
        ax.plot(f_time, scenarios[i, :], color='gray', alpha=0.08, zorder=1)

    ax.fill_between(f_time, quantile_05, quantile_95, alpha=0.2, color='orange', label='90% доверительный интервал прогноза', zorder=2)
    ax.fill_between(f_time, quantile_25, quantile_75, alpha=0.3, color='orange', label='50% доверительный интервал прогноза', zorder=2)
    ax.plot(f_time, quantile_01, color='darkred', linewidth=1.5, linestyle='--', alpha=0.8, label='1%-квантиль (прогноз)', zorder=4)
    ax.plot(f_time, quantile_99, color='darkred', linewidth=1.5, linestyle='--', alpha=0.8, label='99%-квантиль (прогноз)', zorder=4)
    ax.plot(f_time, quantile_50, color='darkred', linewidth=2.5, label='Медиана прогноза', zorder=5)

    ax.axvline(x=0, color='black', linestyle=':', linewidth=1.5, alpha=0.7, zorder=4, label='Старт прогноза (t = 0)')
    ax.axhline(y=S0, color='purple', linestyle='--', linewidth=1, alpha=0.5, label='Цена на старте прогноза S₀')

    ax.set_xlabel('Время t, ч', fontsize=11)
    ax.set_ylabel('Цена S, руб.', fontsize=11)
    method_ru = 'Милштейн' if method == 'milstein' else 'Эйлер'
    ax.set_title(f'{ticker}, {method_ru}', fontsize=12)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.95)
    ax.grid(True, alpha=0.3, linestyle='--')

    metrics_text = (
        f"Медианная доходность (прогноз): {metrics['median_return']:.2%}\n"
        f"μ (дрифт): {mc_result['stats'].get('mu_annual', 0):.4f}\n"
        f"σ (волатильность): {mc_result['stats'].get('sigma_annual', 0):.4f}"
    )
    ax.text(0.02, 0.98, metrics_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontfamily='monospace')

    trading_text = (
        f"Позиция: {position_label}\n"
        f"Вход S₀: {entry_price:.4g} руб.\n"
        f"Медиана прогноза S(T): {exit_price_median:.4g} руб.\n"
        f"Реализовано S(T): {actual_exit_price:.4g} руб.\n"
        f"Реализованный P/L: {actual_pnl_rub:+.4g} руб. ({actual_pnl_pct:+.2%})"
    )
    ax.text(0.02, 0.02, trading_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.9), fontfamily='monospace')

    plt.tight_layout()
    filename = os.path.join(save_dir, f"{ticker}_{method}.png")
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.close(fig)
    gc.collect()

    del quantile_01, quantile_05, quantile_25, quantile_50, quantile_75, quantile_95, quantile_99
    del history, f_time, h_time, scenarios
    return filename


def save_mc_forecast_all_paths_to_file(mc_result: dict, method: str, output_dir: str = ".mcplots"):
    """История + все сценарии (без факта в прогнозе).

    Parameters:
        mc_result: результат одной схемы с заполненным полем scenarios.
        method: 'milstein' или 'euler'.
        output_dir: папка для сохранения.

    Returns:
        Путь к файлу либо None.
    """
    if mc_result["scenarios"] is None:
        return None
    scenarios = mc_result["scenarios"][:, 1:].astype(np.float32)
    n_steps = scenarios.shape[1]
    n_scenarios = scenarios.shape[0]
    train_prices = mc_result["train_prices"]
    ticker = mc_result["ticker"]
    S0 = float(train_prices[-1])
    max_reasonable = S0 * 100
    scenarios = np.minimum(scenarios, max_reasonable)
    history_len = min(HISTORY_HOURS, len(train_prices))
    history = np.asarray(train_prices[-history_len:])
    h_time = np.arange(-history_len, 0, dtype=float)
    f_time = np.arange(0, n_steps, dtype=float)
    fig, ax = plt.subplots(figsize=(14, 7))
    segments = np.zeros((n_scenarios, n_steps, 2), dtype=np.float32)
    segments[:, :, 0] = f_time
    segments[:, :, 1] = scenarios
    lc = LineCollection(segments, cmap=plt.cm.nipy_spectral, array=np.linspace(0, 1, n_scenarios),
                       alpha=0.35, linewidths=0.25)
    ax.add_collection(lc)
    ax.plot(h_time, history, color='#1f77b4', linewidth=2, label='История', alpha=0.95, zorder=5)
    y_min = min(history.min(), scenarios.min())
    y_max = max(history.max(), scenarios.max())
    ax.set_xlim(h_time[0], f_time[-1])
    ax.set_ylim(y_min * 0.98, y_max * 1.02)
    ax.set_xlabel('Время t, ч', fontsize=11)
    ax.set_ylabel('Цена S, руб.', fontsize=11)
    method_ru = 'Милштейн' if method == 'milstein' else 'Эйлер'
    ax.set_title(f'{ticker}, {method_ru}', fontsize=12)
    ax.legend(loc='upper right', fontsize=9, framealpha=0.95)
    ax.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f"{ticker}_{method}_all_paths.png")
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.close(fig)
    gc.collect()
    del segments, lc, scenarios, history
    return filename


def display_two_charts_side_by_side(path_left: str, path_right: str, title_left: str = "Прогноз (квантили и медиана)", title_right: str = "Все сценарии"):
    """Два графика в строку: слева основной, справа все сценарии."""
    try:
        from matplotlib import image as mpimg
        from IPython.display import display
        img_left = mpimg.imread(path_left)
        img_right = mpimg.imread(path_right)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))
        ax1.imshow(img_left)
        ax1.axis("off")
        ax1.set_title(title_left, fontsize=12)
        ax2.imshow(img_right)
        ax2.axis("off")
        ax2.set_title(title_right, fontsize=12)
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    except Exception as e:
        try:
            from IPython.display import display, Image
            display(Image(filename=path_left))
            display(Image(filename=path_right))
        except Exception:
            pass

In [ ]:
# Запуск
if __name__ == "__main__":
    time_start_total = time.time()

    print("=" * 100)
    print("MONTE CARLO ПРОГНОЗ ЦЕН АКЦИЙ MOEX - ПОРТФЕЛЬНЫЕ МЕТРИКИ")
    print("=" * 100)
    print("Конфигурация:")
    print(f" • Прогноз вперёд: {FORECAST_HOURS} ч")
    print(f" • На графиках: 48 ч всего ({HISTORY_HOURS} ч история + {FORECAST_HOURS} ч прогноз)")
    print(f" • История для μ, σ: {LOOKBACK_HOURS} ч (1 год торговых часов)")
    print(f" • Минимум данных на тикер: {LOOKBACK_HOURS + FORECAST_HOURS} ч")
    print(f" • Потоков параллельной обработки: {N_CORES}")
    print(f" • Сценариев MC на каждый тикер: {MC_SCENARIOS:,}")
    print(f" • Порог роста для Long (медиана ≥): {GROWTH_THRESHOLD:.1%}")
    print(f" • Порог падения для Short (медиана ≤): {SHORT_THRESHOLD:.1%}")
    print(f" • Максимум графиков на метод: {MAX_PLOTS_PER_METHOD}")
    print(f" • Разделение графиков по P/L: {'да (pnl_positive / pnl_non_positive)' if SPLIT_PLOTS_BY_PNL else 'нет'}")
    print(f" • Только графики с заработком (P/L факт > 0): {'да' if SHOW_ONLY_PROFITABLE_PLOTS else 'нет'}")
    print(f" • Безрисковая ставка: {RISK_FREE_RATE_ANNUAL*100:.2f}% годовых")

    RESULTS_DIR = os.path.join("results", datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f" • Папка результатов: {RESULTS_DIR}")
    print("=" * 100)
    print()

    loading_stats = LoadingStats()
    warm_up_numba()
    SIM_TIME.clear()

    print("ФАЗА 1: Загрузка данных и расчёт обеих схем на одном снимке данных")
    print("-" * 100)
    print()

    mc_results = {}
    completed = 0

    with ThreadPoolExecutor(max_workers=N_CORES) as executor:
        futures = {
            executor.submit(process_and_analyze, ticker, MC_SCENARIOS): ticker
            for ticker in TICKERS_LIST
        }

        for future in as_completed(futures):
            ticker_name, pair, status, retries, retry_time = future.result()
            completed += 1

            if status == "success" and pair is not None:
                mc_results[ticker_name] = pair

            errors_count = sum(loading_stats.error_counts.values())
            print_progress_line(completed, len(TICKERS_LIST), len(mc_results), errors_count)

            if completed % GC_INTERVAL == 0:
                gc.collect()

    print()
    time_milstein = SIM_TIME["milstein"]
    time_euler = SIM_TIME["euler"]
    loading_snapshot = loading_stats.get_snapshot()
    gc.collect()

    print("ФАЗА 2: Расчёт портфельных метрик")
    print("-" * 100)
    print()

    results_list_milstein = [pair["milstein"] for pair in mc_results.values()]
    results_list_euler = [pair["euler"] for pair in mc_results.values()]

    results_list_long_milstein = [r for r in results_list_milstein if r["metrics"]["median_return"] >= GROWTH_THRESHOLD]
    results_list_short_milstein = [r for r in results_list_milstein if r["metrics"]["median_return"] <= SHORT_THRESHOLD]
    results_list_long_euler = [r for r in results_list_euler if r["metrics"]["median_return"] >= GROWTH_THRESHOLD]
    results_list_short_euler = [r for r in results_list_euler if r["metrics"]["median_return"] <= SHORT_THRESHOLD]

    portfolio_milstein = calculate_portfolio_metrics(results_list_long_milstein, use_short=False)
    portfolio_euler = calculate_portfolio_metrics(results_list_long_euler, use_short=False)
    portfolio_short_milstein = calculate_portfolio_metrics(results_list_short_milstein, use_short=True)
    portfolio_short_euler = calculate_portfolio_metrics(results_list_short_euler, use_short=True)

    print(f"✓ Обработано тикеров (Милштейн): {len(results_list_milstein)}  → Long: {len(results_list_long_milstein)}, Short: {len(results_list_short_milstein)}")
    print(f"✓ Обработано тикеров (Эйлер): {len(results_list_euler)}  → Long: {len(results_list_long_euler)}, Short: {len(results_list_short_euler)}")
    print()

    df_table1, df_table2 = build_portfolio_tables(
        portfolio_milstein, portfolio_euler, time_milstein, time_euler,
        portfolio_short_milstein, portfolio_short_euler
    )
    if df_table1 is not None and df_table2 is not None:
        df_table1.to_csv(os.path.join(RESULTS_DIR, "portfolio_table1.csv"), index=False, encoding="utf-8-sig")
        df_table2.to_csv(os.path.join(RESULTS_DIR, "portfolio_table2.csv"), index=False, encoding="utf-8-sig")
        print("\n" + "=" * 140)
        print("ПОРТФЕЛЬНЫЕ МЕТРИКИ: СРАВНЕНИЕ МЕТОДОВ")
        print("=" * 140)
        print("\nТАБЛИЦА 1: по капиталу — блоки ВСЕГО (Long+Short), Long, Short; вес = доля капитала тикера")
        print("-" * 140)
        print(df_table1.to_string(index=False))
        try:
            from IPython.display import display
            display(df_table1)
        except Exception:
            pass
        print("\nТАБЛИЦА 2: все найденные тикеры по одной штуке, каждая позиция = 1 руб")
        print("-" * 140)
        print(df_table2.to_string(index=False))
        try:
            display(df_table2)
        except Exception:
            pass
        print("=" * 140)
        print(f"Таблицы сохранены: {RESULTS_DIR}/portfolio_table1.csv, {RESULTS_DIR}/portfolio_table2.csv")

    print_loading_statistics(loading_snapshot)

    print_portfolio_summary(
        portfolio_milstein,
        portfolio_euler,
        time_milstein,
        time_euler,
        portfolio_short_milstein,
        portfolio_short_euler,
    )

    print("\nФАЗА 3: Визуализация из данных основного расчёта")
    print("-" * 100)

    for method, results_long, results_short, subdir in (
        ("milstein", results_list_long_milstein, results_list_short_milstein, "mc_plots_milstein"),
        ("euler", results_list_long_euler, results_list_short_euler, "mc_plots_euler"),
    ):
        selected = (results_long + results_short)[:MAX_PLOTS_PER_METHOD]
        plots_dir = os.path.join(RESULTS_DIR, subdir)
        all_paths_dir = os.path.join(plots_dir, "all_paths")
        method_ru = "Милштейн" if method == "milstein" else "Эйлер"
        print(f"Графики ({method_ru}): тикеры итогового расчёта (Long/Short): {len(selected)}")
        saved = 0
        for mc_result in selected:
            try:
                scenarios = regenerate_scenarios(mc_result, MC_SCENARIOS)
                viz_result = dict(mc_result)
                viz_result["scenarios"] = scenarios
                path_left = save_mc_forecast_to_file(viz_result, method, plots_dir)
                path_right = save_mc_forecast_all_paths_to_file(viz_result, method, all_paths_dir)
                if path_left and path_right and os.path.isfile(path_right):
                    display_two_charts_side_by_side(path_left, path_right, "Прогноз (квантили, медиана, факт)", "Все сценарии Монте-Карло")
                elif path_left:
                    try:
                        from IPython.display import display, Image
                        display(Image(filename=path_left))
                    except Exception:
                        pass
                if path_left:
                    saved += 1
                del scenarios, viz_result
                gc.collect()
            except Exception as e:
                if VERBOSE_DIAGNOSTICS:
                    print(f"  [{method_ru}] {mc_result['ticker']}: {e}")
        print(f"✓ Визуализировано пар графиков ({method_ru}): {saved} (слева — квантили/медиана/факт, справа — все сценарии)")
        print()

    time_total = time.time() - time_start_total

    print("=" * 100)
    print("ИТОГОВАЯ СТАТИСТИКА")
    print("=" * 100)
    print(f"Чистое время симуляции (Милштейн): {time_milstein:.2f} сек")
    print(f"Чистое время симуляции (Эйлер): {time_euler:.2f} сек")
    print(f"Общее время: {time_total:.2f} сек ({time_total/60:.2f} мин)")
    print("=" * 100)
    print("\n✅ ПРОГРАММА ЗАВЕРШЕНА УСПЕШНО\n")

In [ ]:
# Время выполнения методов и таблица 1 (по капиталу)
try:
    print("Чистое время симуляции методов (сумма по потокам расчёта):")
    print(f"  Милштейн: {time_milstein:.2f} сек")
    print(f"  Эйлер:    {time_euler:.2f} сек")
    print()
except NameError:
    print("Сначала выполните ячейку с расчётом портфеля (time_milstein/time_euler).")
    print()

if 'df_table1' in dir() and df_table1 is not None:
    print("ТАБЛИЦА 1: по капиталу — блоки ВСЕГО, Long, Short")
    print("-" * 80)
    try:
        from IPython.display import display
        display(df_table1)
    except Exception:
        print(df_table1.to_string(index=False))
else:
    print("Таблица 1: выполните ячейку с расчётом портфеля (df_table1 не найден).")

In [ ]:
# Таблица 2 (1 руб на тикер)
if 'df_table2' in dir() and df_table2 is not None:
    print("ТАБЛИЦА 2: по 1 руб на тикер — блоки ВСЕГО, Long, Short")
    print("-" * 80)
    try:
        from IPython.display import display
        display(df_table2)
    except Exception:
        print(df_table2.to_string(index=False))
else:
    print("Таблица 2: выполните ячейку с расчётом портфеля (df_table2 не найден).")